# 01 Data quality checks

Controllo iniziale su fonti, snapshot, file JSON, tabelle pulite e chiavi di join.


## Parametri modificabili

`RUN_DOWNLOAD` aggiorna i dati grezzi. `RUN_CLEANING` ricostruisce le tabelle pulite. `SNAPSHOT` può essere impostato a una data specifica. `PREVIEW_ROWS` regola le anteprime.


In [ ]:
RUN_DOWNLOAD = True
RUN_CLEANING = True
SNAPSHOT = None
OVERWRITE_RAW = False
PREVIEW_ROWS = 5


In [ ]:
from pathlib import Path
import sys
import pandas as pd
REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from scripts.config import RAW_DATA_DIR, CLEAN_DATA_DIR, OPENPNRR_SOURCES
from scripts.src.download_pnrr_data import download_pnrr_data
from scripts.src.clean_pnrr_projects import build_clean_pnrr_tables
from scripts.utils import latest_snapshot, read_csv


## Fonti configurate

La tabella seguente permette di verificare URL e nomi file senza aprire `config.py`.


In [ ]:
pd.DataFrame([{'source': k, **v} for k, v in OPENPNRR_SOURCES.items()])


## Run e controllo JSON

Ogni fonte deve avere un CSV e un JSON nello snapshot raw. Il manifest segnala download reali o fallback campione.


In [ ]:
raw_dir = download_pnrr_data(snapshot=SNAPSHOT, overwrite=OVERWRITE_RAW) if RUN_DOWNLOAD else RAW_DATA_DIR / latest_snapshot(RAW_DATA_DIR)
clean_dir = build_clean_pnrr_tables(snapshot=raw_dir.name) if RUN_CLEANING else CLEAN_DATA_DIR / latest_snapshot(CLEAN_DATA_DIR)
checks = []
for source_name, source in OPENPNRR_SOURCES.items():
    csv_path = raw_dir / source['raw_filename']
    checks.append({'source': source_name, 'csv_exists': csv_path.exists(), 'json_exists': csv_path.with_suffix('.json').exists()})
pd.DataFrame(checks)


## Anteprima tabelle pulite

Questa cella mostra dimensione e prime righe delle quattro tabelle operative.


In [ ]:
for name in ['projects', 'payments', 'locations', 'territories']:
    df = read_csv(clean_dir / f'{name}.csv')
    print(name, df.shape)
    display(df.head(PREVIEW_ROWS))
